# Analog Circuit Simulation: 555 → LCR Bandpass → 741 Follower

A three-stage analog circuit simulated with symplectic Euler integration:

1. **555 astable oscillator** generates a ~4.8 kHz square wave
2. **Series LCR bandpass filter** (resonant ~5 kHz) strips harmonics, passing the fundamental as a near-sinusoid
3. **741 op-amp voltage follower** buffers the output with realistic slew rate limiting

The simulation runs entirely in Rust — Python controls stepping and visualizes results.

### Setup

```bash
cd crates/minkowski-py
uv sync --all-extras && source .venv/bin/activate
maturin develop --release
```

In [ ]:
import matplotlib.pyplot as plt
import minkowski_py as mk
import numpy as np
import polars as pl

## Run Simulation

200K steps at dt=100ns → 20ms of simulation time. The 555 runs at ~4.8 kHz, so we capture ~96 cycles.

In [ ]:
world = mk.World()
sim = mk.CircuitSim(world, vcc=12.0, dt=1e-7, sample_every=20)
sim.step(200_000)
df = sim.waveform()

print(f"Simulated {sim.total_steps} steps, {sim.sample_count} samples")
print(f"Time range: {df['time'][0] * 1e3:.3f} ms to {df['time'][-1] * 1e3:.3f} ms")
print(df.head())

## Full Waveform

Three node voltages over the entire 20ms simulation. The transient settles within the first ~5ms.

In [ ]:
t_ms = df["time"].to_numpy() * 1e3

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t_ms, df["v_555"].to_numpy(), color="tab:blue", linewidth=0.5)
axes[0].set_ylabel("555 Out (V)")
axes[0].set_title("555 Astable — Square Wave")

axes[1].plot(t_ms, df["v_filter"].to_numpy(), color="tab:orange", linewidth=0.5)
axes[1].set_ylabel("Filter (V)")
axes[1].set_title("LCR Bandpass Output")

axes[2].plot(t_ms, df["v_opamp"].to_numpy(), color="tab:green", linewidth=0.5)
axes[2].set_ylabel("OpAmp (V)")
axes[2].set_xlabel("Time (ms)")
axes[2].set_title("741 Voltage Follower")

plt.tight_layout()
plt.show()

## Steady-State Zoom (Last 2ms)

After the transient dies out, the filter output is a clean sinusoid at the resonant frequency.

In [ ]:
t_max = df["time"][-1]
steady = df.filter(pl.col("time") > t_max - 2e-3)
t_s = steady["time"].to_numpy() * 1e3

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t_s, steady["v_555"].to_numpy(), color="tab:blue", linewidth=0.8)
axes[0].set_ylabel("555 Out (V)")

axes[1].plot(t_s, steady["v_filter"].to_numpy(), color="tab:orange", linewidth=0.8)
axes[1].set_ylabel("Filter (V)")

axes[2].plot(t_s, steady["v_opamp"].to_numpy(), color="tab:green", linewidth=0.8)
axes[2].set_ylabel("OpAmp (V)")
axes[2].set_xlabel("Time (ms)")

fig.suptitle("Steady-State Waveforms (Last 2ms)", fontsize=14)
plt.tight_layout()
plt.show()

## 555 vs Filter Overlay

The LCR bandpass passes the fundamental (~5 kHz) and attenuates harmonics, turning the square wave into a near-sinusoid.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(
    t_s,
    steady["v_555"].to_numpy(),
    color="tab:blue",
    alpha=0.6,
    linewidth=0.8,
    label="555 Output",
)
ax1.set_ylabel("555 Voltage (V)", color="tab:blue")
ax1.set_xlabel("Time (ms)")

ax2 = ax1.twinx()
ax2.plot(
    t_s,
    steady["v_filter"].to_numpy(),
    color="tab:orange",
    linewidth=1.2,
    label="Filter Output",
)
ax2.set_ylabel("Filter Voltage (V)", color="tab:orange")

fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95))
plt.title("Square Wave → Bandpass Filter → Sinusoid")
plt.tight_layout()
plt.show()

## Frequency Analysis (FFT)

Verify the bandpass frequency by computing the FFT of the steady-state filter output.

In [ ]:
# Use last 5ms for cleaner FFT
fft_window = df.filter(pl.col("time") > t_max - 5e-3)
signal = fft_window["v_filter"].to_numpy()
dt_sample = 20 * 1e-7  # dt * sample_every

freqs = np.fft.rfftfreq(len(signal), d=dt_sample)
spectrum = np.abs(np.fft.rfft(signal))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freqs / 1e3, spectrum, color="tab:orange", linewidth=0.8)
ax.set_xlim(0, 20)
ax.set_xlabel("Frequency (kHz)")
ax.set_ylabel("Magnitude")
ax.set_title("FFT of Filter Output")

peak_idx = np.argmax(spectrum[1:]) + 1  # skip DC
peak_freq = freqs[peak_idx]
ax.axvline(peak_freq / 1e3, color="red", linestyle="--", alpha=0.7)
ax.annotate(
    f"Peak: {peak_freq:.0f} Hz",
    xy=(peak_freq / 1e3, spectrum[peak_idx]),
    xytext=(peak_freq / 1e3 + 2, spectrum[peak_idx] * 0.8),
    arrowprops=dict(arrowstyle="->", color="red"),
    color="red",
)

plt.tight_layout()
plt.show()

## Waveform Statistics

In [ ]:
steady_stats = df.filter(pl.col("time") > t_max - 5e-3)

for col in ["v_555", "v_filter", "v_opamp"]:
    s = steady_stats[col]
    vpp = s.max() - s.min()
    rms = np.sqrt((s.to_numpy() ** 2).mean())
    print(f"{col:>10}: mean={s.mean():+.3f}V  Vpp={vpp:.3f}V  rms={rms:.3f}V")

## Explore

- Try `vcc=5.0` for a lower voltage rail — the 741's saturation clips earlier
- Try `vcc=24.0` for higher amplitude
- Decrease `dt` for better accuracy (slower), increase for speed (may become unstable)
- The 741 follower tracks the filter output but slew rate limiting (~0.5 V/µs) becomes visible at higher frequencies